<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/metabolomics/notebooks/02_metabolomics_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Metabolomics Basic Analysis
---

Same 45 patients, same clinical question, different molecules. Yesterday we analysed
1 458 protein groups; today it is **1 073 named metabolites** measured by targeted MRM,
plus **6 pooled quality-control injections** that let us measure our own measurement error.

> **Which serum metabolites separate CRKP sepsis from CSKP sepsis, and what does that say
> about the patient's metabolism?**

The workflow rhymes with the proteomics one, but three steps are genuinely different, and
those differences are the lesson:

| | Proteomics (yesterday) | Metabolomics (today) |
|---|---|---|
| Filtering | completeness only | completeness **+ QC-based** (CV, D-ratio) |
| Imputation | down-shifted normal (missing = low) | half-minimum (missing = below LOQ) |
| Testing | t-test / ANOVA | **ANCOVA**, adjusting for age |
| Annotation | UniProt / GO | **KEGG compounds and pathways** |

In [ ]:
%pip install -q acore vuecore vuegen "pingouin<0.6.0"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from scipy import stats

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

OUT_DIR = Path("metabolomics/report")
SECTIONS = {
    name: OUT_DIR / name
    for name in ["1_quality_control", "2_differential_abundance", "3_enrichment", "4_data"]
}
for path in SECTIONS.values():
    path.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
GROUP_COLOURS = {"Con": "#4C72B0", "CSKP": "#DD8452", "CRKP": "#C44E52", "QC": "#8C8C8C"}
GROUP_ORDER = ["Con", "CSKP", "CRKP"]

## 1. Load and orient the data

In [ ]:
matrix = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv", sep="\t")
annotation = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t").set_index("sample_id")

INFO_COLS = ["metabolite", "formula", "q1_mz", "adduct", "ion_mode"]
metabolite_info = matrix[INFO_COLS].set_index("metabolite")

qc_samples = [c for c in matrix.columns if c.startswith("QC")]
bio_samples = [c for c in matrix.columns if c not in INFO_COLS + qc_samples]

# samples in rows, metabolites in columns — as every statistics function expects
data = matrix.set_index("metabolite")[bio_samples + qc_samples].T
data.index.name = "sample"

print(f"{data.shape[1]} metabolites x {len(bio_samples)} patients + {len(qc_samples)} QC injections")
print(f"annotated with a KEGG compound ID: {annotation['kegg_compound'].notna().sum()}")
data.iloc[:4, :5]

### Mapping the groups

We keep the QC injections in the matrix, we will use them for filtering, and we will drop them before the statistics.

In [ ]:
groups = metadata["group"].reindex(data.index)
groups = groups.fillna("QC")
group_map = {g: groups.index[groups == g].tolist() for g in ["Con", "CSKP", "CRKP", "QC"]}
{k: len(v) for k, v in group_map.items()}

## 2. How bad is the missingness?

In targeted metabolomics a missing value has a clear meaning: the transition was
monitored, and nothing rose above the detection threshold. It is a **left-censored**
value — "less than something" — not an unknown.

In [ ]:
# 1. Missing values bar chart
missing_per_metabolite = data.loc[bio_samples].isna().sum()
frequency = missing_per_metabolite.value_counts().sort_index()

fig1 = px.bar(
    x=frequency.index,
    y=frequency.values,
    labels={"x": "Number of missing values", "y": "Metabolites"},
    title="Missing values per metabolite",
    color_discrete_sequence=["#4C72B0"]
)
fig1.show()

# 2. Intensity distribution histogram
flat_data = data.loc[bio_samples].to_numpy().ravel()
log2_data = np.log2(flat_data[~np.isnan(flat_data)])

fig2 = px.histogram(
    x=log2_data,
    nbins=70,
    labels={"x": "log2 peak area", "y": "count"},
    title="Intensity distribution",
    color_discrete_sequence=["#4C72B0"]
)
fig2.show()

print(f"overall missingness: {data.loc[bio_samples].isna().to_numpy().mean():.2%}")

## 3. Filtering — with the QC samples as our ruler

Three filters, applied in order. All three come from
[`acore.filter_metabolomics`](https://analytics-core.readthedocs.io/).

In [ ]:
from acore import filter_metabolomics as fm

### 3a. The 80 % rule

Keep a metabolite only if it was measured in at least 80 % of the biological samples.
The threshold is a convention, not a law — the point is to avoid testing a metabolite
that exists in five patients.

In [ ]:
step1 = fm.filter_by_missingness(data, method="classic", percent=80, samples=bio_samples)
print(f"80% rule:  {data.shape[1]} -> {step1.shape[1]} metabolites "
      f"({data.shape[1] - step1.shape[1]} removed)")

### 3b. CV filtering — is the signal bigger than the noise?

For each metabolite, compare the coefficient of variation across **pooled QC injections**
(pure technical noise, since they are the same material) with the CV across **patients**
(biology + technical noise). If the QC CV is the larger of the two, the metabolite is
telling us more about the instrument than about the patients.

In [ ]:
step2 = fm.filter_cv(step1, samples=bio_samples, qcs=qc_samples)
print(f"CV filter: {step1.shape[1]} -> {step2.shape[1]} metabolites "
      f"({step1.shape[1] - step2.shape[1]} removed)")

### 3c. D-ratio — the same idea, quantified

The **dispersion ratio** puts a number on it:

$$ D\text{-ratio} = \frac{\sigma_{\text{QC}}}{\sigma_{\text{biological}}} $$

D-ratio is a standardized quality control metric used to evaluate analytical precision and filter out low-quality metabolic features. [1](https://www.sciencedirect.com/science/article/abs/pii/S0165993623000961), [2](https://bioc.r-universe.dev/articles/notame/introduction.html).

Relying solely on CV to flag poor data can be misleading. For example, if a metabolite has a very stable concentration across all biological samples, its technical variance in the QC samples might flag it leading to the accidental removal of a perfectly valid, stable biomarker. [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC10222478/)

A D-ratio near 0 means the variation we see is biological; near 1 means it is measurement
noise. The usual cut-off is 0.5, sometimes stricter. We use the **median absolute
deviation** rather than the standard deviation because raw peak areas are skewed and MAD
ignores extremes.

> ⚙️ This filter is not in `acore` yet, so we write it ourselves.

In [ ]:
import plotly.express as px
import numpy as np

def filter_dratio(data: pd.DataFrame, samples: list, qcs: list, threshold: float = 0.5):
    """Drop features whose QC dispersion is a large fraction of their biological dispersion."""
    def mad(frame: pd.DataFrame) -> pd.Series:
        # Asymptotic Consistency (n -> infty):The constant \(1.4826\) is derived directly 
        # from the standard normal distribution. It is the reciprocal of the third quartile 
        # for an infinite normal population.
        # For small sample size (n < infty), this value can be a bit higher.
        median = frame.median()
        return 1.4826 * (frame - median).abs().median()

    dratio = mad(data.loc[qcs]) / mad(data.loc[samples])
    undefined = dratio.isna() | np.isinf(dratio)
    keep = (dratio <= threshold) & ~undefined
    print(f"D-ratio undefined for {undefined.sum()} metabolites (zero biological dispersion)")
    return data.loc[:, keep], dratio


step3, dratio = filter_dratio(step2, samples=bio_samples, qcs=qc_samples, threshold=0.5)
print(f"D-ratio:   {step2.shape[1]} -> {step3.shape[1]} metabolites "
      f"({step2.shape[1] - step3.shape[1]} removed)")

dratio_cleaned = dratio.replace([np.inf, -np.inf], np.nan).dropna().clip(upper=2)

fig = px.histogram(
    x=dratio_cleaned,
    nbins=60,
    labels={"x": "D-ratio (technical dispersion / biological dispersion)", "y": "Metabolites"},
    title="How much of each signal is noise?",
    color_discrete_sequence=["#4C72B0"]
)

fig.add_vline(
    x=0.5, 
    line_dash="dash", 
    line_color="#C44E52", 
    annotation_text="threshold = 0.5"
)

fig.update_layout(width=700, height=350)
fig.write_image(SECTIONS["1_quality_control"] / "1_dratio.png", scale=2)
fig.show()

In [ ]:
filtering_summary = pd.DataFrame(
    {
        "step": ["raw", "80% rule", "CV filter", "D-ratio ≤ 0.5"],
        "metabolites": [data.shape[1], step1.shape[1], step2.shape[1], step3.shape[1]],
    }
)
filtering_summary["removed"] = -filtering_summary["metabolites"].diff().fillna(0).astype(int)
filtering_summary

✋ **Worth pausing on.** We started with 1 073 metabolites and kept a fraction of them, and
every one of those thresholds was a choice. Filtering is not a technicality that precedes
the analysis; it *is* part of the analysis. Report your thresholds, and check that your
conclusions survive changing them.

## 4. Imputation

Missing here means "below the limit of quantification", so we replace it with something
small: **half the minimum observed value** for that metabolite. 

Half-minimum imputation is a common data-cleaning technique in metabolomics used to handle missing values (often called zeros or NAs) that occur below a mass spectrometer's limit of detection (LOD).

Half-minimum imputation replaces every missing value for a specific metabolite with half of the lowest detected value found for that metabolite across all other samples.

$$ value_{imputed} = \frac{MinimumObservedValue_{metabolite}}{2} $$


In [ ]:
from acore.imputation_analysis import imputation_half_minimum

data_imputed = imputation_half_minimum(data=step3)
print(f"missing before: {step3.isna().to_numpy().mean():.2%}   "
      f"after: {data_imputed.isna().to_numpy().mean():.2%}")
data_log = np.log2(data_imputed)
data_log.iloc[:3, :5]

## 5. Instrumental drift

Metabolomics signal degrades over an analytical run: the column ages, the source gets
dirty, sensitivity drifts. The standard correction fits a **LOESS** curve through the QC
injections *in acquisition order* and rescales the biological samples by the estimated
drift. `acore.drift_correction.run_loess_drift_correction` implements it, and
`run_cpca_drift_correction` offers a components-based alternative.

**Both need the injection order, and MTBLS14016 does not report it.** So we cannot correct
for drift here. Two things reduce the risk: the authors state that all samples were processed 
in a single batch, and our QC CVs were low (median ≈ 0.14), which is not what a badly drifting run looks like.

> 🧬 The practical lesson is about *your* future submissions: **deposit the injection
> order.**

## 6. Normalisation and a first look

Z-scoring each metabolite puts them on a common scale, which is what clustering and PCA
need — otherwise the most abundant compounds dominate purely because they are abundant.

In [ ]:
from acore import normalization

data_norm = normalization.normalize_data(data_log, method="zscore")

# 1. log2 peak areas box plot
fig1 = px.box(
    data_log.T,
    points=False,
    title="log2 peak areas",
    labels={"variable": "Sample", "value": "log2 area"}
)
fig1.update_xaxes(tickangle=90, tickfont=dict(size=5))
fig1.show()

# 2. Normalized peak areas box plot
fig2 = px.box(
    data_norm.T,
    points=False,
    title="after z-scoring each metabolite",
    labels={"variable": "Sample", "value": "z-score"}
)
fig2.update_xaxes(tickangle=90, tickfont=dict(size=5))
fig2.show()

### PCA

This is the plot to look at before anything else. The QC injections are the same material,
so they **must** cluster tightly. If they scatter as widely as the patients, the
experiment has a technical problem and no biological conclusion is safe.

In [ ]:
from acore.decomposition.pca import run_pca

pcs, model = run_pca(data_norm, n_components=3)
explained = model.explained_variance_ratio_ * 100
pcs["group"] = groups.reindex(pcs.index)

pc1_col = pcs.columns[0]
pc2_col = pcs.columns[1]
labels = {
    pc1_col: f"PC1 ({explained[0]:.1f}%)",
    pc2_col: f"PC2 ({explained[1]:.1f}%)"
}

# 1. All injections
pcs_all = pcs[pcs["group"].isin(["Con", "CSKP", "CRKP", "QC"])]
fig1 = px.scatter(
    pcs_all,
    x=pc1_col,
    y=pc2_col,
    color="group",
    color_discrete_map=GROUP_COLOURS,
    title="All injections",
    labels=labels
)
fig1.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
fig1.update_layout(width=700, height=500)
fig1.show()

# 2. Patients only
pcs_patients = pcs[pcs["group"].isin(["Con", "CSKP", "CRKP"])]
fig2 = px.scatter(
    pcs_patients,
    x=pc1_col,
    y=pc2_col,
    color="group",
    color_discrete_map=GROUP_COLOURS,
    title="Patients only",
    labels=labels
)
fig2.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
fig2.update_layout(width=700, height=500)
fig2.write_image(SECTIONS["1_quality_control"] / "2_pca.png", scale=2)
fig2.show()

## 7. Statistics: ANCOVA with age as a covariate

Sepsis outcome depends strongly on age, and age also shapes the serum metabolome
(renal function, muscle mass, medication). If the groups happen to differ in age, a
metabolite that merely tracks age will look like a marker of resistance.

**ANCOVA** (analysis of covariance) is the answer: it compares group means *after*
removing the linear effect of the covariate. Formally, for each metabolite

$$ y = \beta_0 + \beta_1 \cdot \text{group} + \beta_2 \cdot \text{age} + \varepsilon $$

and we test whether $\beta_1 \neq 0$.

First, check whether age is actually unbalanced:

In [ ]:
age_by_group = metadata.groupby("group")["age"].agg(["mean", "std", "min", "max"]).reindex(GROUP_ORDER)
kruskal = stats.kruskal(*[metadata.loc[metadata["group"] == g, "age"] for g in GROUP_ORDER])
print(f"Kruskal-Wallis test for an age difference between groups: p = {kruskal.pvalue:.3f}")
age_by_group

In [ ]:
import acore.differential_regulation as ad

analysis_data = data_norm.loc[bio_samples].copy()
analysis_data.insert(0, "group", metadata.loc[analysis_data.index, "group"].astype(str))
analysis_data.insert(1, "age", metadata.loc[analysis_data.index, "age"].astype(float))

def sanitize_patsy_col_name(col_name):
    """Replace single quotes in column names with a safe string for Patsy."""
    return col_name.replace("'", "__prime__")

# Identify columns that need sanitizing (metabolite names)
metabolite_cols = analysis_data.columns.drop(['group', 'age'])
columns_to_rename = {col: sanitize_patsy_col_name(col) for col in metabolite_cols if "'" in col}

# Apply renaming if any columns need it
if columns_to_rename:
    analysis_data_modified = analysis_data.rename(columns=columns_to_rename)
else:
    analysis_data_modified = analysis_data.copy()

ancova = ad.run_ancova(analysis_data_modified, drop_cols=[], group="group", covariates=["age"])
print("columns returned:", list(ancova.columns))
ancova.head()

In [ ]:
COLUMN_ALIASES = {
    "p-value": "pvalue", "pval": "pvalue",
    "Log2FC": "log2FC", "log2_FC": "log2FC",
    "-log10 p-value": "-log10 pvalue", "FC": "fold_change",
}


def tidy(results: pd.DataFrame) -> pd.DataFrame:
    out = results.rename(columns=COLUMN_ALIASES).copy()
    if "identifier" not in out.columns:
        out = out.rename_axis("identifier").reset_index()
    if "-log10 pvalue" not in out and "pvalue" in out:
        out["-log10 pvalue"] = -np.log10(out["pvalue"])
    return out


ancova = tidy(ancova).sort_values("pvalue")
print(f"metabolites with an overall group effect at p < 0.05: {(ancova['pvalue'] < 0.05).sum()}")
if "padj" in ancova:
    print(f"surviving FDR correction at 5%: {(ancova['padj'] < 0.05).sum()}")
ancova.head(12)

### The pairwise comparison the clinic cares about

The paper compared CRKP with CSKP using variable importance in projection (VIP) values from orthogonal partial least squares discriminant analysis (OPLS-DA) models **OPLS-DA VIP > 1** together with a t-test
p < 0.05, and reported **128 differential metabolites**. VIP (variable importance in
projection) comes from a supervised multivariate model; with 15 samples per group and
1 000 features such models overfit easily, which is why the authors combined VIP with a
univariate test. We will use the univariate test plus a fold-change threshold — simpler,
and easier to defend.

In [ ]:
crkp = data_log.loc[group_map["CRKP"]]
cskp = data_log.loc[group_map["CSKP"]]

pairwise = pd.DataFrame(
    {
        "identifier": data_log.columns,
        "mean_CRKP": crkp.mean().values,
        "mean_CSKP": cskp.mean().values,
        "log2FC": (crkp.mean() - cskp.mean()).values,
        "pvalue": [stats.ttest_ind(crkp[m], cskp[m], equal_var=True).pvalue for m in data_log.columns],
    }
)
pairwise["-log10 pvalue"] = -np.log10(pairwise["pvalue"])
# Benjamini-Hochberg, written out so you can see what it does
ranked = pairwise["pvalue"].rank(method="first")
pairwise["padj"] = (pairwise["pvalue"] * len(pairwise) / ranked).clip(upper=1)
pairwise = pairwise.sort_values("pvalue")

P_CUTOFF, LFC_CUTOFF = 0.05, 0.58
hits = pairwise[(pairwise["pvalue"] < P_CUTOFF) & (pairwise["log2FC"].abs() >= LFC_CUTOFF)]
print(f"CRKP vs CSKP: {len(hits)} metabolites at p < {P_CUTOFF} and |log2FC| >= {LFC_CUTOFF}")
print(f"surviving FDR at 5%: {(pairwise['padj'] < 0.05).sum()}")
pairwise.head(12)

In [ ]:
volcano = pairwise.merge(
    annotation[["metabolite", "class_i", "kegg_compound"]],
    left_on="identifier", right_on="metabolite", how="left",
)
volcano["significant"] = (volcano["pvalue"] < P_CUTOFF) & (volcano["log2FC"].abs() >= LFC_CUTOFF)
volcano["-log10 pvalue"] = -np.log10(volcano["pvalue"])

volcano_fig = px.scatter(
    volcano,
    x="log2FC",
    y="-log10 pvalue",
    color="significant",
    hover_data=["identifier", "class_i", "kegg_compound", "pvalue"],
    color_discrete_map={True: "#C44E52", False: "#9AA5B1"},
    title="CRKP vs CSKP — serum metabolome",
    labels={"log2FC": "log2 fold change (CRKP / CSKP)"},
    width=900,
    height=600,
)
volcano_fig.add_hline(y=-np.log10(P_CUTOFF), line_dash="dot", line_color="grey")
volcano_fig.add_vline(x=LFC_CUTOFF, line_dash="dot", line_color="grey")
volcano_fig.add_vline(x=-LFC_CUTOFF, line_dash="dot", line_color="grey")
volcano_fig.write_json(str(SECTIONS["2_differential_abundance"] / "0_volcano_plot.json"))
volcano_fig

### Which classes of molecule are moving?

Unlike proteins, metabolites come with a chemical taxonomy. Asking whether our hits are
concentrated in one class is a quick, interpretable form of enrichment.

In [ ]:
class_counts = pd.DataFrame(
    {
        "hits": volcano[volcano["significant"]]["class_i"].value_counts(),
        "measured": volcano["class_i"].value_counts(),
    }
).dropna(subset=["measured"]).fillna(0)
class_counts["fraction of class that is a hit"] = class_counts["hits"] / class_counts["measured"]
class_counts.sort_values("fraction of class that is a hit", ascending=False).head(10)

## 8. Comparison with the published results

In [ ]:
published = pd.read_csv(f"{BASE_URL}/metabolomics/data/published_dems.tsv", sep="\t")
published_crkp = published.query("comparison == 'CRKP_vs_KP'")
published_significant = set(
    published_crkp.loc[
        (published_crkp["vip"] > 1) & (published_crkp["p_value"] < 0.05), "metabolite"
    ]
)
ours = set(hits["identifier"])

print(f"published differential metabolites (VIP > 1, p < 0.05) : {len(published_significant)}")
print(f"ours (p < 0.05, |log2FC| >= 0.58)                      : {len(ours)}")
print(f"in both                                                : {len(ours & published_significant)}")

In [ ]:
shared = sorted(ours & published_significant)
if shared:
    reference = published_crkp.set_index("metabolite")
    ours_indexed = hits.set_index("identifier")
    
    plot_df = pd.DataFrame({
        "published_lfc": reference.loc[shared, "log2_fold_change"],
        "our_lfc": ours_indexed.loc[shared, "log2FC"]
    })
    
    fig = px.scatter(
        plot_df,
        x="published_lfc",
        y="our_lfc",
        title="Published vs recomputed effect sizes",
        labels={
            "published_lfc": "published log2 fold change",
            "our_lfc": "our log2 fold change"
        },
        color_discrete_sequence=["#4C72B0"],
        width=520,
        height=500
    )
    
    fig.update_traces(marker=dict(size=8, opacity=0.75))
    
    fig.add_shape(
        type="line",
        x0=-2, y0=-2,
        x1=2, y1=2,
        line=dict(dash="dot", color="grey")
    )
    
    fig.show()

## 9. Pathway enrichment with KEGG

Metabolite identifiers are the hard part of metabolomics, and here they are handed to us:
the vendor library provides **KEGG compound IDs** and **KEGG pathway maps** for most
annotated compounds. That lets us go from a list of molecule names to a statement about
metabolism — and, crucially, it uses the *same* pathway vocabulary as the proteins, which
is what makes the multi-omics integration this afternoon possible.

In [ ]:
pathway_annotation = (
    annotation.loc[annotation["kegg_map"].notna(), ["metabolite", "kegg_map"]]
    .assign(pathway=lambda d: d["kegg_map"].str.split(","))
    .explode("pathway")
    .assign(pathway=lambda d: d["pathway"].str.strip())
    .rename(columns={"metabolite": "identifier"})[["identifier", "pathway"]]
    .drop_duplicates()
)

# Human-readable names for the ko identifiers, taken from the published enrichment table.
kegg_names = (
    pd.read_csv(f"{BASE_URL}/metabolomics/data/published_kegg_dems.tsv", sep="\t")
    .dropna(subset=["ko_id"])
    .drop_duplicates("ko_id")
    .set_index("ko_id")["kegg_pathway"]
)
pathway_annotation["pathway_name"] = pathway_annotation["pathway"].map(kegg_names)

print(f"{pathway_annotation['identifier'].nunique()} metabolites mapped to "
      f"{pathway_annotation['pathway'].nunique()} KEGG pathways")
pathway_annotation.head()

> 💻 To fetch the mapping live from KEGG instead — recommended for a real project, since
> pathways change — use `acore.io.kegg`:
>
> ```python
> from acore.io.kegg import link_kegg_batch, parse_compound_pathway_mapping
> raw = link_kegg_batch("pathway", annotation["kegg_compound"].dropna().unique())
> mapping = parse_compound_pathway_mapping(raw)
> ```
> The equivalent REST call is <https://rest.kegg.jp/link/compound/pathway>.

### The enrichment test

For each pathway we build a 2×2 table — in the hit list or not, in the pathway or not —
and apply Fisher's exact test. The **background** is every metabolite that survived
filtering, *not* the whole of KEGG: we can only discover what our assay could measure.

In [ ]:
import acore.enrichment_analysis

regulation = pairwise.rename(columns={"pvalue": "pvalue"}).copy()
regulation["group1"] = "CRKP"
regulation["group2"] = "CSKP"

enriched = acore.enrichment_analysis.run_up_down_regulation_enrichment(
    regulation_data=regulation,
    annotation=pathway_annotation,
    identifier="identifier",
    annotation_col="pathway",
    pval_col="pvalue",
    lfc_cutoff=LFC_CUTOFF,
    min_detected_in_set=2,
    correction_alpha=0.25,
)
if len(enriched):
    label = "terms" if "terms" in enriched.columns else enriched.columns[0]
    enriched["pathway_name"] = enriched[label].map(kegg_names)
print(f"{len(enriched)} enriched pathways")
enriched.head(15)

In [ ]:
if len(enriched):
    pcol = "padj" if "padj" in enriched.columns else "pvalue"
    top = enriched.nsmallest(12, pcol).sort_values(pcol, ascending=False).copy()
    top["pathway_labels"] = top["pathway_name"].fillna(top[label]).astype(str).str.slice(0, 55)
    top["neg_log10_p"] = -np.log10(top[pcol])

    fig = px.bar(
        top,
        x="neg_log10_p",
        y="pathway_labels",
        orientation="h",
        title="Enriched KEGG pathways, CRKP vs CSKP",
        labels={"neg_log10_p": "-log10 adjusted p-value", "pathway_labels": ""},
        color_discrete_sequence=["#4C72B0"],
        width=900,
        height=480,
    )
    
    fig.write_image(str(SECTIONS["3_enrichment"] / "1_kegg_enrichment.png"), scale=2)
    fig.show()

### The pathway the paper points at

The integrated proteome + metabolome analysis highlighted **cysteine and methionine
metabolism** (hsa00270 / ko00270) and the **one-carbon pool by folate** (ko00670),
connected through the protein **MAT2B**. Let us look at the metabolites we measured in
that first pathway.

In [ ]:
target_pathways = ["ko00270", "ko00670"]
in_target = pathway_annotation[pathway_annotation["pathway"].isin(target_pathways)]
target_metabolites = [m for m in in_target["identifier"].unique() if m in data_log.columns]
print(f"measured metabolites in {target_pathways}: {len(target_metabolites)}")
print(target_metabolites)

In [ ]:
if target_metabolites:
    long = (
        data_log[target_metabolites]
        .loc[bio_samples]
        .join(metadata[["group"]])
        .melt(id_vars="group", var_name="metabolite", value_name="log2 peak area")
    )
    n_panels = min(len(target_metabolites), 8)
    plot_data = long[long["metabolite"].isin(target_metabolites[:n_panels])].copy()
    plot_data["metabolite_short"] = plot_data["metabolite"].astype(str).str.slice(0, 28)

    fig = px.box(
        plot_data,
        x="group",
        y="log2 peak area",
        color="group",
        facet_col="metabolite_short",
        facet_col_wrap=4,
        category_orders={"group": GROUP_ORDER},
        color_discrete_map=GROUP_COLOURS,
        points=False,
        title="Cysteine/methionine and one-carbon metabolites",
        width=900,
        height=600
    )
    
    # Make y-axes independent (sharey=False) and remove facet variable prefix from titles
    fig.update_yaxes(matches=None)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    
    fig.write_image(str(SECTIONS["3_enrichment"] / "2_target_pathway_metabolites.png"), scale=2)
    fig.show()

🧬 **Why would methionine metabolism matter in a resistant infection?** Methionine feeds
S-adenosylmethionine (SAM), the cell's universal methyl donor, which links to DNA and
protein methylation, to polyamine synthesis, and — through the transsulfuration pathway —
to glutathione, the main intracellular antioxidant. A septic patient under oxidative
stress drains that pool. Bacteria also compete for methionine and its precursors, and
*K. pneumoniae* carbapenemase-producing strains carry additional metabolic burdens.
So a shift here is biologically plausible from at least three directions — which is a
reason to be interested, and also a reason to be careful: plausible stories can be told
about almost any pathway after the fact.

## 10. Save results and build the report

In [ ]:
data_log.to_csv(SECTIONS["4_data"] / "1_filtered_log2_metabolite_matrix.csv")
data_norm.to_csv(SECTIONS["4_data"] / "2_zscored_metabolite_matrix.csv")
filtering_summary.to_csv(SECTIONS["1_quality_control"] / "3_filtering_summary.csv", index=False)
ancova.to_csv(SECTIONS["2_differential_abundance"] / "1_ancova_age_adjusted.csv", index=False)
pairwise.to_csv(SECTIONS["2_differential_abundance"] / "2_ttest_CRKP_vs_CSKP.csv", index=False)
hits.to_csv(SECTIONS["2_differential_abundance"] / "3_significant_metabolites.csv", index=False)
if len(enriched):
    enriched.to_csv(SECTIONS["3_enrichment"] / "0_kegg_enrichment.csv", index=False)

!find metabolomics/report -type f | sort

In [ ]:
!vuegen --directory metabolomics/report --report_type html

## 🔗 Where this goes next

This afternoon we will use the two data types -- proteomics and metabolomics, and stop treating them as two
datasets:

- **Multi-omics I** — integrate them (SNF, MOFA) and ask whether the *combination*
  separates patients that neither layer separates alone.
- **Multi-omics II** — build a network in which proteins and metabolites are neighbours,
  and read the biology off its structure.

## 📚 Further reading

- acore recipes for metabolomics — <https://analytics-core.readthedocs.io/>
- Broadhurst *et al.* (2018) *Guidelines and considerations for the use of system
  suitability and quality control samples in mass spectrometry assays.* Metabolomics 14:72.
  — the source of the D-ratio and QC practice used above.
- Chong *et al.* (2019) *MetaboAnalyst 4.0.* Nucleic Acids Res — the toolset the paper used.
- Wishart *et al.* (2022) *HMDB 5.0: the Human Metabolome Database.* Nucleic Acids Res 50:D622–D631.